# MoCo — Momentum Contrast for Unsupervised Visual Representation Learning
**He et al., CVPR 2020**

**Category:** `03-Self-Supervised-Learning / 02-ContrastiveLearning`

SimCLR needs giant batches (4096+) for enough negatives. MoCo decouples **dictionary size from batch size** using:
- A **FIFO queue** of 65,536 encoded keys (negatives)
- A **momentum encoder** updated slowly via EMA — keeps keys consistent across time

| | SimCLR | MoCo |
|---|---|---|
| Negatives | Current batch (2N-2) | Queue (65K) |
| Batch size needed | 4096+ | 256 |
| Key encoder update | Backprop (end-to-end) | EMA (no grad) |


<img src="../figures/moco_arch.png" width="800"/>

*MoCo: query encoder (backprop) + momentum key encoder (EMA) + FIFO queue of 4096 negatives.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import copy, math
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Step 1: Two-View Augmentation (same as SimCLR)

Same augmentation pipeline as SimCLR: RandomCrop + ColorJitter + GaussianBlur.

Unlike SimCLR, the two views go to **different encoders** — the query view goes to `f_q` (trained by backprop), the key view goes to `f_k` (updated by EMA only, never receives gradients directly).

In [ ]:
class MoCoAugmentation:
    def __init__(self, size=32):
        self.transform = T.Compose([
            T.RandomResizedCrop(size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.GaussianBlur(kernel_size=3),
            T.ToTensor(),
            T.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010]),
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x)

class TwoViewDataset(Dataset):
    def __init__(self, aug):
        self.base = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=None)
        self.aug  = aug
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, _ = self.base[idx]
        return self.aug(img)


## Step 2: MoCo Model
### Dual Encoder Architecture
- **Query encoder** `f_q`: updated by backprop normally
- **Key encoder** `f_k`: updated by EMA of f_q — NOT by gradient

$$\theta_k \leftarrow m \cdot \theta_k + (1 - m) \cdot \theta_q$$

`m = 0.999` → key encoder changes glacially → all 65K keys in queue remain consistent


In [ ]:
class MoCo(nn.Module):
    def __init__(self, base_encoder=torchvision.models.resnet18,
                 dim=128, K=4096, m=0.999, T=0.07):
        super().__init__()
        self.K = K   # queue size
        self.m = m   # momentum coefficient
        self.T = T   # temperature

        # Query encoder
        self.encoder_q = base_encoder(weights=None)
        self.encoder_q.fc = nn.Sequential(
            nn.Linear(self.encoder_q.fc.in_features, dim))

        # Key encoder (copy of query, no grad)
        self.encoder_k = copy.deepcopy(self.encoder_q)
        for p in self.encoder_k.parameters():
            p.requires_grad = False

        # Queue: (dim, K)  — stored as columns
        self.register_buffer('queue', F.normalize(
            torch.randn(dim, K), dim=0))
        self.register_buffer('queue_ptr', torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update(self):
        """EMA update of key encoder."""
        for p_q, p_k in zip(self.encoder_q.parameters(),
                             self.encoder_k.parameters()):
            p_k.data = self.m * p_k.data + (1.0 - self.m) * p_q.data

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        """Enqueue new keys, dequeue oldest (FIFO)."""
        N = keys.shape[0]
        ptr = int(self.queue_ptr)
        # Wrap-around: fill end then start
        if ptr + N <= self.K:
            self.queue[:, ptr:ptr + N] = keys.T
        else:
            end = self.K - ptr
            self.queue[:, ptr:]  = keys[:end].T
            self.queue[:, :N-end]= keys[end:].T
        self.queue_ptr[0] = (ptr + N) % self.K

    def forward(self, x_q, x_k):
        # Query: with gradient
        q = F.normalize(self.encoder_q(x_q), dim=1)          # (N, dim)

        # Key: no gradient
        with torch.no_grad():
            self._momentum_update()
            k = F.normalize(self.encoder_k(x_k), dim=1)      # (N, dim)

        # Positive logits: (N, 1)
        l_pos = torch.einsum('nd,nd->n', [q, k]).unsqueeze(-1)

        # Negative logits: (N, K)
        l_neg = torch.mm(q, self.queue.clone().detach())

        # InfoNCE loss
        logits = torch.cat([l_pos, l_neg], dim=1) / self.T   # (N, 1+K)
        labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
        loss = F.cross_entropy(logits, labels)

        self._dequeue_and_enqueue(k)
        return loss, q, k


## Step 3: Training

Key difference from SimCLR: the queue decouples dictionary size from batch size.
- Batch of 64 images → 64 query vectors vs **4096 key vectors** in the queue
- Each training step: encode new keys → enqueue → dequeue oldest
- The EMA momentum (m=0.999) keeps the key encoder changing slowly → consistent keys across the queue lifespan

> **Why EMA instead of a copy?** If the key encoder updated every step like the query encoder, old keys in the queue would be encoded by a different (outdated) network — inconsistent representations. Slow EMA keeps the queue consistent.

In [ ]:
BATCH_SIZE = 128
EPOCHS     = 10
LR         = 0.03
MOMENTUM   = 0.9
WEIGHT_DECAY = 1e-4

aug    = MoCoAugmentation()
loader = DataLoader(TwoViewDataset(aug), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=2, drop_last=True)

model     = MoCo(K=4096, m=0.999, T=0.07).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                             momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

losses = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = []
    for (x_q, x_k) in loader:
        x_q, x_k = x_q.to(device), x_k.to(device)
        loss, _, _ = model(x_q, x_k)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    scheduler.step()
    avg = sum(epoch_loss)/len(epoch_loss)
    losses.append(avg)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg:.4f}')

import os; os.makedirs('saved', exist_ok=True)
torch.save(model.state_dict(), 'saved/moco.pt')


## Step 4: Linear Evaluation

Same protocol as SimCLR: freeze the **query encoder** `f_q`, train a linear classifier on top.

MoCo should match SimCLR accuracy at much smaller batch sizes (64 vs 4096). This is the key result — the queue gives you 4096 effective negatives without 4096 images in the batch.

In [ ]:
model.load_state_dict(torch.load('saved/moco.pt', map_location=device))
for p in model.encoder_q.parameters():
    p.requires_grad = False

# Get encoder output dim
dim_feat = model.encoder_q[:-1] if hasattr(model.encoder_q, '__getitem__') else None
# Use encoder without the final projection layer
backbone = torchvision.models.resnet18(weights=None)
backbone.fc = nn.Identity()
backbone.to(device)
# Copy weights from query encoder's backbone layers
backbone_state = {k.replace('encoder_q.', ''): v
                  for k, v in model.state_dict().items()
                  if k.startswith('encoder_q.') and 'fc' not in k}
backbone.load_state_dict(backbone_state, strict=False)
for p in backbone.parameters():
    p.requires_grad = False

linear = nn.Linear(512, 10).to(device)
opt_lin = torch.optim.Adam(linear.parameters(), lr=1e-3)

eval_tf = T.Compose([T.ToTensor(),
    T.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010])])
train_ld = DataLoader(torchvision.datasets.CIFAR10('./data', True,  eval_tf, download=True), 256, shuffle=True)
test_ld  = DataLoader(torchvision.datasets.CIFAR10('./data', False, eval_tf, download=True), 256)

for epoch in range(5):
    backbone.eval(); linear.train()
    for imgs, labels in train_ld:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            h = backbone(imgs)
        loss = F.cross_entropy(linear(h), labels)
        opt_lin.zero_grad(); loss.backward(); opt_lin.step()

correct = total = 0
backbone.eval(); linear.eval()
with torch.no_grad():
    for imgs, labels in test_ld:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = linear(backbone(imgs)).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
print(f'MoCo Linear Eval: {correct/total*100:.2f}%')
